# Fase 2 - Modelado no supervisado de anomalías

Laboratorio reproducible para comparar enfoques no supervisados sin modificar el pipeline productivo ni guardar datos sensibles en salidas.

Este notebook debe ejecutarse localmente con artefactos generados por Fase 1. No contiene resultados precalculados.

## Objetivo

- Revisar el baseline actual de reglas + Isolation Forest.
- Comparar modelos no supervisados.
- Evaluar estabilidad, sensibilidad y acuerdo entre modelos.
- Preparar top-N de alertas para validación experta.
- Definir criterios para promover un modelo a producción.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import OneClassSVM

ROOT = Path.cwd().resolve()
FEATURES_PATH = ROOT / "data" / "processed" / "10_features_y_scores_diarios.csv"
ANOMALIES_PATH = ROOT / "data" / "processed" / "11_anomalias_consumo.csv"
OUTPUT_DIR = ROOT / "reports" / "fase_2_experimentos"

RANDOM_SEEDS = [7, 13, 29, 42, 101]
CONTAMINATIONS = [0.01, 0.02, 0.035, 0.05, 0.08]
TOP_N = 50

## Carga de datos

Este notebook usa salidas locales ignoradas por Git. Si no existen, ejecute primero el pipeline de Fase 1.

In [ ]:
if not FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"No existe {FEATURES_PATH}. Ejecute primero: "
        "python -m granjas_anomalias.cli run --config config/project.yml"
    )

features = pd.read_csv(FEATURES_PATH, parse_dates=["fecha"])
features.shape

## Calidad de datos

Revisar estructura, nulos y rangos antes de entrenar modelos.

In [ ]:
quality_summary = pd.DataFrame(
    {
        "dtype": features.dtypes.astype(str),
        "missing_pct": features.isna().mean().round(4),
        "n_unique": features.nunique(dropna=True),
    }
)
quality_summary.sort_values(["missing_pct", "n_unique"], ascending=[False, True]).head(40)

## Variables candidatas

No incluir identificadores de orden, lote ni documento SAP como variables del modelo. Se conservan solo para trazabilidad posterior.

In [ ]:
MODEL_FEATURES = [
    "consumo_g_ave_dia_real",
    "brecha_consumo_pct_dia",
    "consumo_robust_z_28",
    "cambio_consumo_pct_dia",
    "consumo_promedio_7d",
    "consumo_std_7d",
    "mortalidad_por_1000",
    "ratio_produccion_vs_estandar",
    "edad_semana",
    "fases_activas_dia",
]

missing_model_features = [column for column in MODEL_FEATURES if column not in features.columns]
if missing_model_features:
    raise ValueError(f"Faltan variables candidatas: {missing_model_features}")

eligible = (
    features["aves_disponibles"].gt(0)
    & features["consumo_real_kg_dia"].ge(0)
    & features["dias_desde_inicio"].ge(7)
)

X = features.loc[eligible, MODEL_FEATURES].replace([np.inf, -np.inf], np.nan)
X.shape

## Segmentación

Revisar comportamiento por ciclo, caseta, edad y fase antes de comparar modelos.

In [ ]:
segment_columns = [
    column
    for column in ["cycle_id", "caseta", "edad_semana", "fase_alimento_principal"]
    if column in features.columns
]

segment_summary = (
    features.groupby(segment_columns, dropna=False)
    .agg(
        registros=("fecha", "size"),
        consumo_promedio=("consumo_real_kg_dia", "mean"),
        score_promedio=("score_anomalia", "mean"),
        alertas=("es_anomalia", "sum"),
    )
    .reset_index()
)
segment_summary.head(20)

## Prevención de fuga temporal

Definir cortes por fecha o ciclo completo. No usar split aleatorio simple como validación principal.

In [ ]:
ordered = features.loc[eligible].sort_values("fecha").copy()
cutoff = ordered["fecha"].quantile(0.70)
train_mask = ordered["fecha"].le(cutoff)
review_mask = ordered["fecha"].gt(cutoff)

X_train = ordered.loc[train_mask, MODEL_FEATURES].replace([np.inf, -np.inf], np.nan)
X_review = ordered.loc[review_mask, MODEL_FEATURES].replace([np.inf, -np.inf], np.nan)
(X_train.shape, X_review.shape)

## Reproducción del baseline actual

El baseline actual usa Isolation Forest con imputación mediana y escalamiento robusto.

In [ ]:
def anomaly_score_from_decision(decision_values: np.ndarray) -> np.ndarray:
    decision = -decision_values
    minimum = float(np.nanmin(decision))
    maximum = float(np.nanmax(decision))
    return 100 * (decision - minimum) / max(maximum - minimum, 1e-12)


baseline_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
        (
            "model",
            IsolationForest(
                n_estimators=400,
                contamination=0.035,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

baseline_model.fit(X_train)
baseline_scores = anomaly_score_from_decision(baseline_model.decision_function(X_review))
baseline_flags = baseline_model.predict(X_review) == -1
pd.Series(baseline_scores).describe()

## Comparación de modelos

Comparar Isolation Forest, Local Outlier Factor y One-Class SVM. Ajustar parámetros con cuidado para evitar exceso de alertas.

In [ ]:
def fit_iforest(seed: int, contamination: float):
    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
            (
                "model",
                IsolationForest(
                    n_estimators=400,
                    contamination=contamination,
                    random_state=seed,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    model.fit(X_train)
    return model


def fit_lof(contamination: float):
    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
            (
                "model",
                LocalOutlierFactor(
                    n_neighbors=35,
                    contamination=contamination,
                    novelty=True,
                ),
            ),
        ]
    )
    model.fit(X_train)
    return model


def fit_ocsvm(nu: float):
    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
            ("model", OneClassSVM(kernel="rbf", gamma="scale", nu=nu)),
        ]
    )
    model.fit(X_train)
    return model

## Sensibilidad de parámetros y estabilidad entre semillas

In [ ]:
experiment_rows = []
score_table = pd.DataFrame(index=ordered.loc[review_mask].index)

for contamination in CONTAMINATIONS:
    for seed in RANDOM_SEEDS:
        model = fit_iforest(seed=seed, contamination=contamination)
        score = anomaly_score_from_decision(model.decision_function(X_review))
        flag = model.predict(X_review) == -1
        name = f"iforest_c{contamination}_s{seed}"
        score_table[name] = score
        experiment_rows.append(
            {
                "modelo": "IsolationForest",
                "contamination": contamination,
                "seed": seed,
                "alertas": int(flag.sum()),
                "score_p95": float(np.percentile(score, 95)),
            }
        )

experiments = pd.DataFrame(experiment_rows)
experiments

## Acuerdo entre modelos

In [ ]:
lof_model = fit_lof(contamination=0.035)
ocsvm_model = fit_ocsvm(nu=0.035)

agreement = ordered.loc[review_mask].copy()
agreement["score_iforest"] = baseline_scores
agreement["flag_iforest"] = baseline_flags
agreement["score_lof"] = anomaly_score_from_decision(lof_model.decision_function(X_review))
agreement["flag_lof"] = lof_model.predict(X_review) == -1
agreement["score_ocsvm"] = anomaly_score_from_decision(ocsvm_model.decision_function(X_review))
agreement["flag_ocsvm"] = ocsvm_model.predict(X_review) == -1
agreement["acuerdo_modelos"] = agreement[["flag_iforest", "flag_lof", "flag_ocsvm"]].sum(axis=1)

agreement[["fecha", "cycle_id", "caseta", "score_iforest", "score_lof", "score_ocsvm", "acuerdo_modelos"]].sort_values(
    ["acuerdo_modelos", "score_iforest"], ascending=False
).head(TOP_N)

## Perturbaciones sintéticas

Aplicar cambios controlados sobre una copia para revisar sensibilidad. No usar estos casos como etiquetas reales.

In [ ]:
synthetic = ordered.loc[review_mask].copy()
synthetic_features = synthetic[MODEL_FEATURES].copy()

if not synthetic_features.empty:
    perturbed = synthetic_features.copy()
    sample_index = perturbed.index[: min(10, len(perturbed))]
    perturbed.loc[sample_index, "consumo_g_ave_dia_real"] *= 1.8
    perturbed.loc[sample_index, "cambio_consumo_pct_dia"] = 0.8
    perturbed_scores = anomaly_score_from_decision(baseline_model.decision_function(perturbed))
else:
    perturbed_scores = np.array([])

pd.Series(perturbed_scores).describe()

## Top-N para revisión experta

Exportar solo si el equipo confirma que el destino es local y seguro. No publicar con datos reales.

In [ ]:
review_columns = [
    column
    for column in [
        "fecha",
        "cycle_id",
        "caseta",
        "edad_semana",
        "fase_alimento_principal",
        "consumo_real_kg_dia",
        "consumo_estandar_kg_dia",
        "score_anomalia",
        "severidad",
        "motivo_anomalia",
        "score_iforest",
        "score_lof",
        "score_ocsvm",
        "acuerdo_modelos",
    ]
    if column in agreement.columns
]

top_n_review = agreement.sort_values(
    ["acuerdo_modelos", "score_iforest"], ascending=False
).head(TOP_N)[review_columns]

top_n_review

## Recomendación

Completar después de revisar estabilidad, sensibilidad, acuerdo entre modelos y feedback experto.

Criterios mínimos para promover un modelo:

- aporta valor sobre reglas;
- alertas explicables;
- estabilidad temporal y entre semillas;
- volumen revisable;
- validación experta documentada;
- sin fuga temporal ni dependencia de identificadores sensibles.